# GDF 02 - Lakehouse Router (Optimized + XLSX Audit)

**Default Lakehouse: Configuration**

Preserves the existing routing rule:
`<Target Lakehouse>/Files/<Country>/<Category>/<Data Type>/<File>`

Shared mapping remains at:
`Files/UL_Router/Configuration/Mapping.xlsx`

Performance changes:
- Opens Mapping.xlsx once.
- Builds all lookup dictionaries once.
- Scans staging once.
- Creates/lists each target folder once, instead of repeated `exists/ls/mkdir` calls for every file.
- ABFS-root validation is optional and disabled by default.
- Keeps safe temporary-copy then atomic overwrite/move behavior.


In [ ]:
PipelineRunId = "MANUAL_RUN"
MAPPING_FILE = "/lakehouse/default/Files/UL_Router/Configuration/Mapping.xlsx"
SOURCE_ROOT = "Files/UL_Router/GDF/ReadyForSharePoint1"
CATEGORY_SHEET = "Category"
LAKEHOUSE_SHEET = "LakeHouse mapping"
SUBCATEGORY_COLUMN = "SUB CATEGORY"
DASHBOARD_CATEGORY_COLUMN = "Dashboard Category"
CATEGORY_KEY_COLUMN = "Lakehouse_Key"
LAKEHOUSE_NAME_COLUMN = "Dashboard / Lakehouse"
ABFS_COLUMN = "ABFS Path"
DRY_RUN = False
DELETE_SOURCE_AFTER_SUCCESS = False
VALIDATE_ABFS_ROOTS = False
TEST_COUNTRY = ""
AUDIT_DIR = "Files/UL_Router/GDF/Audit/Router"

SOURCE_STABILITY_RETRIES = 12
SOURCE_STABILITY_DELAY_SECONDS = 10
MAPPING_COPY_RETRIES = 12
MAPPING_COPY_RETRY_DELAY_SECONDS = 10


In [ ]:
import os,re,json,uuid,time,tempfile,shutil
from collections import defaultdict
from datetime import datetime,timezone
from pathlib import Path
import pandas as pd
import notebookutils

run_id=(str(PipelineRunId).strip() if PipelineRunId and str(PipelineRunId).strip()!='MANUAL_RUN'
        else 'manual_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
router_run_id=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:8]

def normalize(v): return re.sub(r'[^a-z0-9]','',str(v).lower()) if v is not None else ''
def clean_text(v):
    if v is None:return ''
    return re.sub(r'[ \t]+',' ',str(v).replace('\xa0',' ').strip())
def clean_abfs(v): return str(v or '').replace('\r','').replace('\n','').strip().rstrip('/')
def join_path(base,*parts): return base.rstrip('/')+'/'+ '/'.join(str(p).strip('/') for p in parts if p is not None and str(p)!='')

def walk_remote(root):
    def _walk(current,parts):
        for item in notebookutils.fs.ls(current):
            name=item.name.rstrip('/')
            if item.isDir: yield from _walk(item.path,parts+[name])
            elif item.isFile: yield item,parts+[name]
    yield from _walk(root,[])

def find_col(columns,names):
    wanted={normalize(n) for n in names}
    return next((c for c in columns if normalize(c) in wanted),None)

# Materialize the shared Mapping.xlsx locally via OneLake API.
_mapping_remote="Files/UL_Router/Configuration/Mapping.xlsx"
_mapping_local=Path(tempfile.gettempdir())/f"GDF_Mapping_{run_id}.xlsx"
_last=""
for _attempt in range(1,MAPPING_COPY_RETRIES+1):
    try:
        if _mapping_local.exists(): _mapping_local.unlink()
        _ok=notebookutils.fs.cp(_mapping_remote,"file:"+str(_mapping_local),recurse=False)
        if _ok and _mapping_local.exists() and _mapping_local.stat().st_size>0:
            MAPPING_FILE=str(_mapping_local)
            break
        _last="copy returned false or local file empty"
    except Exception as _e:
        _last=str(_e)
    if _attempt<MAPPING_COPY_RETRIES:
        time.sleep(MAPPING_COPY_RETRY_DELAY_SECONDS)
else:
    raise RuntimeError(f"Mapping workbook could not be materialized locally: {_last}")
print('PipelineRunId:',run_id)
print('Mapping:',MAPPING_FILE)
print('Source:',SOURCE_ROOT)


In [ ]:
# Open workbook once and resolve sheet names case/spacing-insensitively.
xls=pd.ExcelFile(MAPPING_FILE,engine='openpyxl')
def resolve_sheet(desired):
    nd=normalize(desired)
    hit=next((s for s in xls.sheet_names if normalize(s)==nd),None)
    if not hit: raise ValueError(f"Worksheet '{desired}' not found. Available: {xls.sheet_names}")
    return hit

category_df=xls.parse(resolve_sheet(CATEGORY_SHEET),dtype=str).fillna('')
lakehouse_df=xls.parse(resolve_sheet(LAKEHOUSE_SHEET),dtype=str).fillna('')
country_sheet=next((s for s in xls.sheet_names if normalize(s)==normalize('Country')),None)
country_df=xls.parse(country_sheet,dtype=str).fillna('') if country_sheet else pd.DataFrame()

required_cat={SUBCATEGORY_COLUMN,DASHBOARD_CATEGORY_COLUMN,CATEGORY_KEY_COLUMN}
required_lh={LAKEHOUSE_NAME_COLUMN,ABFS_COLUMN}
if required_cat-set(category_df.columns): raise ValueError(f'Missing Category columns: {sorted(required_cat-set(category_df.columns))}')
if required_lh-set(lakehouse_df.columns): raise ValueError(f'Missing LakeHouse mapping columns: {sorted(required_lh-set(lakehouse_df.columns))}')

# Reporting dictionaries from codes, if available.
country_code_map={}
if not country_df.empty:
    cc=find_col(country_df.columns,['Cont','Country Code','CountryCode']); cn=find_col(country_df.columns,['Country'])
    if cc and cn:
        country_code_map={str(k).strip().upper():str(v).strip() for k,v in zip(country_df[cc],country_df[cn]) if str(k).strip() and str(v).strip()}
cat_code_col=find_col(category_df.columns,['CATEGORY']); cat_name_col=find_col(category_df.columns,['CATEGORY NAME'])
category_code_name={}
if cat_code_col and cat_name_col:
    category_code_name={str(k).strip().upper():str(v).strip() for k,v in zip(category_df[cat_code_col],category_df[cat_name_col]) if str(k).strip() and str(v).strip()}

subcategory_to_key={}; dashboard_to_key={}; conflicts=[]
for _,row in category_df.iterrows():
    key=clean_text(row[CATEGORY_KEY_COLUMN])
    if not key: continue
    for sub in [clean_text(x) for x in re.split(r'[\r\n]+',str(row[SUBCATEGORY_COLUMN] or '')) if clean_text(x)]:
        n=normalize(sub)
        if n in subcategory_to_key and subcategory_to_key[n]!=key: conflicts.append(f"SUB CATEGORY '{sub}' conflicts")
        else: subcategory_to_key[n]=key
    dash=clean_text(row[DASHBOARD_CATEGORY_COLUMN])
    if dash:
        n=normalize(dash)
        if n in dashboard_to_key and dashboard_to_key[n]!=key: conflicts.append(f"Dashboard Category '{dash}' conflicts")
        else: dashboard_to_key[n]=key
if conflicts: raise RuntimeError('Conflicting category mappings: '+ '; '.join(conflicts))

lakehouse_to_abfs={}
for _,row in lakehouse_df.iterrows():
    name=clean_text(row[LAKEHOUSE_NAME_COLUMN]); abfs=clean_abfs(row[ABFS_COLUMN])
    if not name and not abfs: continue
    if not name or not abfs: raise ValueError(f'Incomplete LakeHouse mapping row: {name} / {abfs}')
    if not abfs.lower().startswith('abfss://'): raise ValueError(f'Invalid ABFS for {name}: {abfs}')
    n=normalize(name)
    if n in lakehouse_to_abfs and lakehouse_to_abfs[n]['abfs']!=abfs: raise RuntimeError(f'Duplicate ABFS mapping for {name}')
    lakehouse_to_abfs[n]={'display_name':name,'abfs':abfs}

def get_target_from_key(key):
    t=lakehouse_to_abfs.get(normalize(key))
    if not t: raise ValueError(f"Lakehouse_Key '{key}' does not match Dashboard / Lakehouse")
    return t

def resolve_category(category):
    n=normalize(category)
    if n in lakehouse_to_abfs: return {'resolution':'DIRECT_DASHBOARD_LAKEHOUSE','lakehouse_key':normalize(lakehouse_to_abfs[n]['display_name']),'target':lakehouse_to_abfs[n]}
    if n in subcategory_to_key:
        key=subcategory_to_key[n]; return {'resolution':'SUB_CATEGORY','lakehouse_key':key,'target':get_target_from_key(key)}
    if n in dashboard_to_key:
        key=dashboard_to_key[n]; return {'resolution':'DASHBOARD_CATEGORY','lakehouse_key':key,'target':get_target_from_key(key)}
    t=lakehouse_to_abfs.get(n)
    if t:return {'resolution':'NORMALIZED_KEY','lakehouse_key':category,'target':t}
    raise ValueError(f"No mapping found for source category '{category}'")

# Validate keys locally (cheap); remote ABFS checks only when explicitly enabled.
for key in set(subcategory_to_key.values())|set(dashboard_to_key.values()): get_target_from_key(key)
if VALIDATE_ABFS_ROOTS:
    bad=[]
    for rec in lakehouse_to_abfs.values():
        try:
            if not notebookutils.fs.exists(rec['abfs']): bad.append(f"{rec['display_name']} -> {rec['abfs']}")
        except Exception as e: bad.append(f"{rec['display_name']} -> {e}")
    if bad: raise RuntimeError('ABFS destination validation failed: '+ '; '.join(bad))
print('Category mappings:',len(subcategory_to_key),'Lakehouses:',len(lakehouse_to_abfs))


In [ ]:
if not notebookutils.fs.exists(SOURCE_ROOT):
    raise FileNotFoundError(f'Source staging root not found: {SOURCE_ROOT}')

# Wait until the visible staged-file count is stable across two checks.
_prev=-1
source_files=[]
for _attempt in range(1,SOURCE_STABILITY_RETRIES+1):
    current=list(walk_remote(SOURCE_ROOT))
    _count=len(current)
    print(f"Source readiness attempt {_attempt}: {_count} files visible")
    if _count>0 and _count==_prev:
        source_files=current
        break
    _prev=_count
    if _attempt<SOURCE_STABILITY_RETRIES:
        time.sleep(SOURCE_STABILITY_DELAY_SECONDS)
else:
    raise RuntimeError(f'Source staging did not stabilize under {SOURCE_ROOT}. Last visible count={_prev}')

routes=[]; validation_errors=[]
for fi,parts in source_files:
    if len(parts)!=4:
        validation_errors.append({'FileName':parts[-1] if parts else '', 'Reason':f"Unexpected hierarchy: {'/'.join(parts)}"})
        continue
    country,category,data_type,filename=parts
    if TEST_COUNTRY and normalize(country)!=normalize(TEST_COUNTRY): continue
    try:
        resolved=resolve_category(category); target=resolved['target']
        target_folder=join_path(target['abfs'],'GDF',country,category,data_type)
        identity=Path(filename).stem.upper().split('_')[0]
        cc=''; catcode=''
        if identity.startswith('UB') and len(identity)>=4:
            cc=identity[2:4]; catcode=identity[4:]
        routes.append({
            'source':fi.path,'source_size':fi.size,'country_code':cc,'country':country,
            'category_code':catcode,'category_name':category_code_name.get(catcode,''),'category':category,
            'data_type':data_type,'filename':filename,'resolution':resolved['resolution'],
            'lakehouse_key':resolved['lakehouse_key'],'target_lakehouse':target['display_name'],
            'target_folder':target_folder,'target_file':join_path(target_folder,filename)
        })
    except Exception as e:
        validation_errors.append({'FileName':filename,'Reason':str(e),'CountryName':country,'DashboardCategory':category})

if validation_errors:
    local=Path('/lakehouse/default')/AUDIT_DIR; local.mkdir(parents=True,exist_ok=True)
    p=local/f'GDF_Router_{run_id}.xlsx'
    with pd.ExcelWriter(p,engine='openpyxl') as w:
        pd.DataFrame([{'PipelineRunId':run_id,'RouterRunId':router_run_id,'Status':'FAILED','Planned':len(routes),'ValidationErrors':len(validation_errors)}]).to_excel(w,index=False,sheet_name='Run Summary')
        pd.DataFrame(validation_errors).to_excel(w,index=False,sheet_name='Validation Errors')
    raise RuntimeError(f'{len(validation_errors)} routing validation error(s). See {p}')

print('Validated files:',len(routes))


In [ ]:
# Optimized copy: one mkdir/list per target folder, not per file.
groups=defaultdict(list)
for r in routes: groups[r['target_folder']].append(r)
results=[]
for folder,items in groups.items():
    if not DRY_RUN:
        notebookutils.fs.mkdirs(folder)
        try:
            existing={x.name.rstrip('/') for x in notebookutils.fs.ls(folder) if x.isFile}
        except Exception:
            existing=set()
    else:
        existing=set()

    for r in items:
        operation='REPLACE_EXACT_FILENAME' if r['filename'] in existing else 'COPY_NEW_FILE'
        base={
            'PipelineRunId':run_id,'RouterRunId':router_run_id,'EventTimeUTC':datetime.now(timezone.utc).isoformat(),
            'FileName':r['filename'],'Step':'ROUTE_TO_LAKEHOUSE','CountryCode':r['country_code'],
            'CountryName':r['country'],'CategoryCode':r['category_code'],'CategoryName':r['category_name'],
            'DashboardCategory':r['category'],'DataType':r['data_type'],'LakehouseKey':r['lakehouse_key'],
            'TargetLakehouse':r['target_lakehouse'],'TargetPath':r['target_file'],'Action':operation,
            'SourcePath':r['source'],'SourceSize':r['source_size'],'Resolution':r['resolution']
        }
        if DRY_RUN:
            results.append({**base,'Status':'DRY_RUN','Reason':''}); continue
        temp=join_path(folder,f".incoming_{uuid.uuid4().hex}_{r['filename']}")
        try:
            if not notebookutils.fs.cp(r['source'],temp,recurse=False): raise RuntimeError('notebookutils.fs.cp returned False')
            if not notebookutils.fs.mv(temp,r['target_file'],create_path=True,overwrite=True): raise RuntimeError('notebookutils.fs.mv returned False')
            existing.add(r['filename'])
            if DELETE_SOURCE_AFTER_SUCCESS: notebookutils.fs.rm(r['source'],recurse=False)
            results.append({**base,'Status':'SUCCESS','Reason':''})
        except Exception as e:
            try:
                if notebookutils.fs.exists(temp): notebookutils.fs.rm(temp,recurse=False)
            except: pass
            results.append({**base,'Status':'FAILED','Reason':str(e)})

rdf=pd.DataFrame(results)


In [ ]:
local=Path('/lakehouse/default')/AUDIT_DIR
local.mkdir(parents=True,exist_ok=True)
audit_path=local/f'GDF_Router_{run_id}.xlsx'
success=rdf[rdf['Status'].astype(str).str.upper().eq('SUCCESS')] if not rdf.empty else pd.DataFrame()
failed=rdf[rdf['Status'].astype(str).str.upper().eq('FAILED')] if not rdf.empty else pd.DataFrame()
affected=(success[['TargetLakehouse']].drop_duplicates().sort_values('TargetLakehouse') if not success.empty else pd.DataFrame(columns=['TargetLakehouse']))
summary=pd.DataFrame([{
    'PipelineRunId':run_id,'RouterRunId':router_run_id,'GeneratedAtUTC':datetime.now(timezone.utc).isoformat(),
    'Status':'FAILED' if not failed.empty else ('DRY_RUN' if DRY_RUN else 'SUCCESS'),
    'Planned':len(routes),'Success':len(success),'Failed':len(failed),
    'NewFiles':int((success['Action']=='COPY_NEW_FILE').sum()) if not success.empty else 0,
    'ReplacedExactFilename':int((success['Action']=='REPLACE_EXACT_FILENAME').sum()) if not success.empty else 0,
    'AffectedLakehouses':len(affected),'SourceRoot':SOURCE_ROOT,'MappingFile':MAPPING_FILE
}])
with pd.ExcelWriter(audit_path,engine='openpyxl') as w:
    summary.to_excel(w,index=False,sheet_name='Run Summary')
    rdf.to_excel(w,index=False,sheet_name='File Log')
    affected.to_excel(w,index=False,sheet_name='Affected Lakehouses')
    failed.to_excel(w,index=False,sheet_name='Failures')

payload=summary.iloc[0].to_dict(); payload['AuditPath']=str(audit_path); payload['AffectedLakehouseNames']=affected['TargetLakehouse'].tolist()
print(json.dumps(payload,indent=2,default=str))
if not failed.empty: raise RuntimeError(f'{len(failed)} file(s) failed. See {audit_path}')
notebookutils.notebook.exit(json.dumps(payload,default=str))
